In [1]:
# %pip install pymupdf python-docx
# %pip install spacy
# %pip install spacy-transformers
# For the high-accuracy Transformer model (F1 ~90%)
# %pip install https://huggingface.co/opennyaiorg/en_legal_ner_trf/resolve/main/en_legal_ner_trf-any-py3-none-any.whl

In [2]:
import fitz  # PyMuPDF
import re
import os
from docx import Document
import tkinter as tk
from tkinter import filedialog

In [3]:
def legal_cleaning(text):
    """
    Enhanced cleaning for Indian Law Reports: 
    Removes margin ladders (A-H), page headers, and numbers 
    while preserving structure for Legal-BERT.
    """
    # 1. Remove "Alphabet Ladders" (Standalone A-H on their own lines)
    # These appear in the margins of court reporters [cite: 1, 6, 209]
    text = re.sub(r'^\s*[A-H]\s*$', '', text, flags=re.MULTILINE)
    
    # 2. Remove Page Headers/Footers [cite: 109, 110]
    # Targeted patterns found in your specific document
    headers = [
        r'\[2020\]\s*1\s*S\.C\.R\.', 
        r'SUPREME\s*COURT\s*REPORTS',
        r'UNION\s*OF\s*INDIA\s*v\.\s*RELIANCE\s*COMMUNICATION.*',
        r'S\.\s*RAVINDRA\s*BHAT,\s*J\.'
    ]
    for pattern in headers:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Remove Standalone Page Numbers [cite: 108, 138, 189]
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Standard Cleaning Logic (Preserved from your original code)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r' +', ' ', text)
    text = "".join(char for char in text if char.isprintable() or char in ['\n', '\t'])
    
    # Final trim to ensure no trailing blank lines from the removed ladders
    text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])
    
    return text.strip()

In [4]:
def process_document():
    # 2. Interactive File Selection
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True) 
    
    # FIX 1: Define the allowed file types in a list of tuples
    file_path = filedialog.askopenfilename(
        title="Layer 1: Select Legal Document (PDF or Word)",
        filetypes=[
            ("PDF files", "*.pdf"),
            ("Word documents", "*.docx"),
            ("All files", "*.*")
        ]
    )

    if not file_path:
        print("Selection cancelled.")
        return None

    # FIX 2: Corrected the syntax for getting the extension
    # It was: os.path.splitext(file_path).[1]lower()
    ext = os.path.splitext(file_path)[1].lower() 
    
    print(f"Extracting: {os.path.basename(file_path)}...")

    # Get the file name and remove .pdf from the og filename
    fileName = str(os.path.basename(file_path)).replace(".pdf", "")

    # 3. Text Extraction
    raw_text = ""
    try:
        if ext == ".pdf":
            with fitz.open(file_path) as doc:
                raw_text = "\n".join([page.get_text("text") for page in doc])
        elif ext == ".docx":
            doc = Document(file_path)
            raw_text = "\n".join([para.text for para in doc.paragraphs])
        
        # 4. Pre-processing & Cleaning
        cleaned_text = legal_cleaning(raw_text)

        # 5. Output for later stages
        folder_name = "Text_Extracts"
        os.makedirs(folder_name, exist_ok=True)   # Creates folder if it doesn't exis   
        output_file = os.path.join(folder_name, f"{fileName}.txt")

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"Success! Cleaned text saved to: {output_file}")
        print(f"Character count: {len(cleaned_text)}")
        return cleaned_text

    except Exception as e:
        print(f"Extraction error: {str(e)}")
        return None

In [5]:
# process_document()

In [6]:
# # Import the segmentation file
# from segmentation import nlp

In [7]:
# # Run Layer 1
# cleaned_data = process_document() # This triggers the file picker
# if cleaned_data:
#     doc = nlp(cleaned_data)

In [8]:
# print(f"--- Extracted Legal Entities ---")
# for ent in doc.ents:
#     print(f"[{ent.label_}]: {ent.text}")

In [9]:
cleaned_data = process_document() # This triggers the file picker


Extracting: 2020_1_1_7_EN.pdf...
Success! Cleaned text saved to: Text_Extracts\2020_1_1_7_EN.txt
Character count: 15220


In [10]:
from caller import extract_legal_details

In [ ]:
# Get the extracted facts, evidence and basic IPCs from the text file
result = extract_legal_details(cleaned_data)
print(result)

```python
{
    "FACTS": [
        "The Union of India invited bids for spectrum auctions via Notice Inviting Applications (NIA) in 2013 and 2015.",
        "The respondents (Reliance Communication Ltd and Reliance Telecom Ltd) were successful bidders but faced financial constraints and strategic debt restructuring.",
        "The respondents defaulted on deferred spectrum charges amounting to Rs. 774.25 crores due in May 2018.",
        "The Union encashed bank guarantees totaling Rs. 908.91 crores, resulting in an excess encashment of Rs. 134.66 crores beyond the admitted liability.",
        "Despite the respondents providing fresh bank guarantees for subsequent installments (Rs. 774.25 crores), the Union refused to refund the excess amount, citing subsequent defaults and procedural objections.",
        "The Telecom Disputes Settlement and Appellate Tribunal (TDSAT) ordered the Union to refund Rs. 104.34 crores after adjusting Rs. 30.33 crores against other charges.",
        "The 